# Teacher prompt

> The prompt that has Claude write the training message for each triage record

In [ ]:
#| default_exp teacher

The teacher is a large model, Claude, that writes the message for each synthetic triage record. Its messages are the training targets for the SLM, so this prompt defines what a good message is. It follows the structure of [write_prose](https://github.com/AnswerDotAI/aai-coding/blob/main/aai_coding/write_prose.py): rules, good examples, one bad message with its mistakes marked, a numbered list of failure modes, and plain words to use. The SLM never sees this prompt. Fine-tuning builds the behaviour into it, so the prompt's length only costs the teacher, and prompt caching covers most of that.

The failure modes are shared. The code checks in [`checks`](02_checks.ipynb) cover some of them, and the judges and the annotation app will use the same list for the rest, until expert error analysis replaces it.

In [ ]:
#| export
import json
from fastllm.chat import AsyncChat, contents
import hashlib
from datetime import datetime, timezone

In [ ]:
#| hide
import re
from fastcore.test import *
from edgeslm.checks import *
from edgeslm.synth import *
from edgeslm.coffee import *

In [ ]:
#| hide
from IPython.display import Markdown
def md_table(rows, cols):
    "A Markdown table of `cols` from each dict in `rows`"
    return '\n'.join([' | '.join(cols), ' | '.join(['---']*len(cols))] + [' | '.join(str(r[c]) for c in cols) for r in rows])
def example_md(e):
    "An example's record items and its message, as Markdown"
    rec = e['record']; head = {k: v for k, v in rec.items() if k != 'items'}
    items = '\n'.join(json.dumps(it, ensure_ascii=False) for it in rec['items'])
    quote = '\n'.join('> ' + l if l else '>' for l in e['message'].split('\n'))
    return f"```json\n{json.dumps(head, ensure_ascii=False)}\n{items}\n```\n\n{quote}"

## Rules

The rules restate the message rules of the [scope](design/scope.ipynb) for the teacher. The triage contract keeps them simple: items arrive most severe first, and `since` is a phrase to copy.

In [ ]:
#| export
CONTEXT = """You write short WhatsApp messages to coffee farmers. Each message turns one triage record into actions. The record comes from monitoring code, which has already decided which problems matter, how severe they are, and what the data shows about their cause. Only some plants carry a sensor, and each sensor stands for the plants around it."""

RULES = """- Write one paragraph per item, in the record's order.
- Start each paragraph with the item's marker (👀 early, ⚠️ moderate, 🚨 severe), then its location exactly as written in the record, then a colon.
- After the colon, say in plain words what is wrong and when it started, then what to do. Say only what the record reports, and never name a cause or a part it does not report, such as a valve, a battery or a radio link. Never write likely, may, might or probably.
- Say when the problem started with the record's `since` phrase, as it is. When `soil_water` has the same phrase, say it once, such as "have had no water and shown water stress for a week or more".
- After the location, refer to it with words such as "this line", "this plot" or "this plant". Never repeat the location or part of it.
- Write no digits outside the locations: no readings, counts or dates. Speak of monitored plants. For a single node, say "a monitored plant" or "this plant".
- Every cluster or widespread item has an `amount` word: all, almost all, most or some. Use it as it is, even when it is all, as in "all monitored plants on this line".
- For water stress, `soil_water` says whether water still reaches the soil. When it is `none` with a phrase, say that no water has reached the soil since then, and point to the irrigation where the pattern does: whether water reaches this plant for one node, the line from its start to the last plant for a cluster, the irrigation schedule and the water supply for the whole farm.
- When `soil_water` is `arriving`, say that the soil still gets water. For one node or a cluster, suggest checking the plants for heat damage, pests or root problems. For the whole farm, the heat is the cause, so suggest longer or more frequent irrigation.
- For equipment faults, `fault` says what the data shows. For `missing data`, say that the sensors have sent no data and that the plants they cover are not monitored. For `implausible values`, say that the `sensor` gives values that look wrong. For `camera view changed`, say that the camera no longer sees the plant's leaves.
- For faults, point to where to look: the node's power and connection for one node, the power supply its nodes share for a cluster, the gateway and its power supply for the whole farm.
- For severe items, say that the action is for today.
- Be plain, friendly and direct. No greeting, no sign-off, no explanation of the monitoring.
- Keep the whole message under 600 characters."""

The context and rules, as the teacher reads them:

In [ ]:
Markdown(f'{CONTEXT}\n\n{RULES}')

You write short WhatsApp messages to coffee farmers. Each message turns one triage record into actions. The record comes from monitoring code, which has already decided which problems matter, how severe they are, and what the data shows about their cause. Only some plants carry a sensor, and each sensor stands for the plants around it.

- Write one paragraph per item, in the record's order.
- Start each paragraph with the item's marker (👀 early, ⚠️ moderate, 🚨 severe), then its location exactly as written in the record, then a colon.
- After the colon, say in plain words what is wrong and when it started, then what to do. Say only what the record reports, and never name a cause or a part it does not report, such as a valve, a battery or a radio link. Never write likely, may, might or probably.
- Say when the problem started with the record's `since` phrase, as it is. When `soil_water` has the same phrase, say it once, such as "have had no water and shown water stress for a week or more".
- After the location, refer to it with words such as "this line", "this plot" or "this plant". Never repeat the location or part of it.
- Write no digits outside the locations: no readings, counts or dates. Speak of monitored plants. For a single node, say "a monitored plant" or "this plant".
- Every cluster or widespread item has an `amount` word: all, almost all, most or some. Use it as it is, even when it is all, as in "all monitored plants on this line".
- For water stress, `soil_water` says whether water still reaches the soil. When it is `none` with a phrase, say that no water has reached the soil since then, and point to the irrigation where the pattern does: whether water reaches this plant for one node, the line from its start to the last plant for a cluster, the irrigation schedule and the water supply for the whole farm.
- When `soil_water` is `arriving`, say that the soil still gets water. For one node or a cluster, suggest checking the plants for heat damage, pests or root problems. For the whole farm, the heat is the cause, so suggest longer or more frequent irrigation.
- For equipment faults, `fault` says what the data shows. For `missing data`, say that the sensors have sent no data and that the plants they cover are not monitored. For `implausible values`, say that the `sensor` gives values that look wrong. For `camera view changed`, say that the camera no longer sees the plant's leaves.
- For faults, point to where to look: the node's power and connection for one node, the power supply its nodes share for a cluster, the gateway and its power supply for the whole farm.
- For severe items, say that the action is for today.
- Be plain, friendly and direct. No greeting, no sign-off, no explanation of the monitoring.
- Keep the whole message under 600 characters.

## Good examples

Five records from a training seed, each from a different tuple, and one chosen for a sensor with implausible values, with messages written by hand:

In [ ]:
#| export
EXAMPLES = [
    dict(record={"farm": "Sítio Recanto", "date": "2026-07-23", "nodes": 40, "items": [{"problem": "water stress", "location": "Block F, drip line F2", "pattern": "cluster", "severity": "early", "since": "since yesterday", "soil_water": "none since Tuesday", "nodes_flagged": 3, "nodes_in_group": 6, "amount": "some"}, {"problem": "water stress", "location": "Block C, drip line C3", "pattern": "cluster", "severity": "early", "since": "since today", "soil_water": "none since yesterday", "nodes_flagged": 7, "nodes_in_group": 7, "amount": "all"}]},
         message='''👀 Block F, drip line F2: some monitored plants on this line have shown first signs of water stress since yesterday, and no water has reached their soil since Tuesday. Check the line from its start to the last plant.

👀 Block C, drip line C3: all monitored plants on this line have shown first signs of water stress since today, and no water has reached their soil since yesterday. Check that the line delivers water when irrigation runs.'''),
    dict(record={"farm": "Fazenda Boa Vista", "date": "2026-05-16", "nodes": 21, "items": [{"problem": "equipment fault", "location": "Plot 3, row 7", "pattern": "one node", "severity": "severe", "since": "since Tuesday", "fault": "missing data"}, {"problem": "water stress", "location": "Plot 1, row 1", "pattern": "one node", "severity": "moderate", "since": "since Thursday", "soil_water": "arriving"}]},
         message='''🚨 Plot 3, row 7: the sensors at this plant have sent no data since Tuesday, so it is not monitored. Check the node's power and connection today.

⚠️ Plot 1, row 1: a monitored plant has shown water stress since Thursday, although its soil still gets water. Check the plant for heat damage, pests or root problems.'''),
    dict(record={"farm": "Sítio Recanto", "date": "2026-08-04", "nodes": 6, "items": [{"problem": "equipment fault", "location": "whole farm", "pattern": "widespread", "severity": "severe", "since": "since Thursday", "fault": "missing data", "nodes_flagged": 6, "nodes_in_group": 6, "amount": "all"}]},
         message='''🚨 Whole farm: all sensors have sent no data since Thursday, so the farm is not monitored. Check the gateway and its power supply today.'''),
    dict(record={"farm": "Fazenda Cachoeira", "date": "2026-08-03", "nodes": 25, "items": [{"problem": "equipment fault", "location": "Plot 1, row 2", "pattern": "one node", "severity": "early", "since": "since today", "fault": "implausible values", "sensor": "soil moisture"}]},
         message='''👀 Plot 1, row 2: the soil moisture sensor at this plant has given values that look wrong since today. Check that it sits firmly in the soil and is not damaged.'''),
    dict(record={"farm": "Sítio Bela Vista", "date": "2026-05-19", "nodes": 47, "items": [{"problem": "water stress", "location": "Block C, row 4", "pattern": "one node", "severity": "early", "since": "since today", "soil_water": "none since Sunday"}, {"problem": "water stress", "location": "Block D, drip line D3", "pattern": "cluster", "severity": "early", "since": "since today", "soil_water": "arriving", "nodes_flagged": 6, "nodes_in_group": 7, "amount": "almost all"}]},
         message='''👀 Block C, row 4: a monitored plant has shown first signs of water stress since today, and no water has reached its soil since Sunday. Check that water reaches this plant when irrigation runs.

👀 Block D, drip line D3: almost all monitored plants on this line have shown first signs of water stress since today, although their soil still gets water. Check the plants for heat damage, pests or root problems.'''),
    dict(record={"farm": "Fazenda São João", "date": "2026-08-17", "nodes": 18, "items": [{"problem": "water stress", "location": "whole farm", "pattern": "widespread", "severity": "early", "since": "since today", "soil_water": "none since Saturday", "nodes_flagged": 17, "nodes_in_group": 18, "amount": "almost all"}, {"problem": "equipment fault", "location": "Block D, row 24", "pattern": "one node", "severity": "early", "since": "since today", "fault": "camera view changed"}]},
         message='''👀 Whole farm: almost all monitored plants have shown first signs of water stress since today, and no water has reached their soil since Saturday. Check that irrigation runs as scheduled and that the water supply works.

👀 Block D, row 24: the camera at this plant has not seen its leaves since today. Check that it points at the plant and that nothing blocks its view.''')]

In [ ]:
Markdown('\n\n---\n\n'.join(map(example_md, EXAMPLES)))

```json
{"farm": "Sítio Recanto", "date": "2026-07-23", "nodes": 40}
{"problem": "water stress", "location": "Block F, drip line F2", "pattern": "cluster", "severity": "early", "since": "since yesterday", "soil_water": "none since Tuesday", "nodes_flagged": 3, "nodes_in_group": 6, "amount": "some"}
{"problem": "water stress", "location": "Block C, drip line C3", "pattern": "cluster", "severity": "early", "since": "since today", "soil_water": "none since yesterday", "nodes_flagged": 7, "nodes_in_group": 7, "amount": "all"}
```

> 👀 Block F, drip line F2: some monitored plants on this line have shown first signs of water stress since yesterday, and no water has reached their soil since Tuesday. Check the line from its start to the last plant.
>
> 👀 Block C, drip line C3: all monitored plants on this line have shown first signs of water stress since today, and no water has reached their soil since yesterday. Check that the line delivers water when irrigation runs.

---

```json
{"farm": "Fazenda Boa Vista", "date": "2026-05-16", "nodes": 21}
{"problem": "equipment fault", "location": "Plot 3, row 7", "pattern": "one node", "severity": "severe", "since": "since Tuesday", "fault": "missing data"}
{"problem": "water stress", "location": "Plot 1, row 1", "pattern": "one node", "severity": "moderate", "since": "since Thursday", "soil_water": "arriving"}
```

> 🚨 Plot 3, row 7: the sensors at this plant have sent no data since Tuesday, so it is not monitored. Check the node's power and connection today.
>
> ⚠️ Plot 1, row 1: a monitored plant has shown water stress since Thursday, although its soil still gets water. Check the plant for heat damage, pests or root problems.

---

```json
{"farm": "Sítio Recanto", "date": "2026-08-04", "nodes": 6}
{"problem": "equipment fault", "location": "whole farm", "pattern": "widespread", "severity": "severe", "since": "since Thursday", "fault": "missing data", "nodes_flagged": 6, "nodes_in_group": 6, "amount": "all"}
```

> 🚨 Whole farm: all sensors have sent no data since Thursday, so the farm is not monitored. Check the gateway and its power supply today.

---

```json
{"farm": "Fazenda Cachoeira", "date": "2026-08-03", "nodes": 25}
{"problem": "equipment fault", "location": "Plot 1, row 2", "pattern": "one node", "severity": "early", "since": "since today", "fault": "implausible values", "sensor": "soil moisture"}
```

> 👀 Plot 1, row 2: the soil moisture sensor at this plant has given values that look wrong since today. Check that it sits firmly in the soil and is not damaged.

---

```json
{"farm": "Sítio Bela Vista", "date": "2026-05-19", "nodes": 47}
{"problem": "water stress", "location": "Block C, row 4", "pattern": "one node", "severity": "early", "since": "since today", "soil_water": "none since Sunday"}
{"problem": "water stress", "location": "Block D, drip line D3", "pattern": "cluster", "severity": "early", "since": "since today", "soil_water": "arriving", "nodes_flagged": 6, "nodes_in_group": 7, "amount": "almost all"}
```

> 👀 Block C, row 4: a monitored plant has shown first signs of water stress since today, and no water has reached its soil since Sunday. Check that water reaches this plant when irrigation runs.
>
> 👀 Block D, drip line D3: almost all monitored plants on this line have shown first signs of water stress since today, although their soil still gets water. Check the plants for heat damage, pests or root problems.

---

```json
{"farm": "Fazenda São João", "date": "2026-08-17", "nodes": 18}
{"problem": "water stress", "location": "whole farm", "pattern": "widespread", "severity": "early", "since": "since today", "soil_water": "none since Saturday", "nodes_flagged": 17, "nodes_in_group": 18, "amount": "almost all"}
{"problem": "equipment fault", "location": "Block D, row 24", "pattern": "one node", "severity": "early", "since": "since today", "fault": "camera view changed"}
```

> 👀 Whole farm: almost all monitored plants have shown first signs of water stress since today, and no water has reached their soil since Saturday. Check that irrigation runs as scheduled and that the water supply works.
>
> 👀 Block D, row 24: the camera at this plant has not seen its leaves since today. Check that it points at the plant and that nothing blocks its view.

Every example passes the code checks:

In [ ]:
test_eq([[k for k, v in check(e['record'], e['message']).items() if not v] for e in EXAMPLES], [[]]*len(EXAMPLES))

## A bad message

This message for the first example makes most of the mistakes the teacher must avoid. Each bracketed number refers to a failure mode in the next section, and the last line marks the item it leaves out:

In [ ]:
#| export
BAD = """Hello! [1][5] Our monitoring system has analysed your farm today. [2]

⚠️ [6] Block F, drip line F2: most [11] nodes [3] are flagged [3] for water stress since 21 July [4] because the valve is stuck [14]. Block F, drip line F2 [8] may need attention. [9] Please check your farm. [10]

Have a great day! [1]

[7: the item at Block C, drip line C3 is missing]"""

In [ ]:
Markdown(f'```text\n{BAD}\n```')

```text
Hello! [1][5] Our monitoring system has analysed your farm today. [2]

⚠️ [6] Block F, drip line F2: most [11] nodes [3] are flagged [3] for water stress since 21 July [4] because the valve is stuck [14]. Block F, drip line F2 [8] may need attention. [9] Please check your farm. [10]

Have a great day! [1]

[7: the item at Block C, drip line C3 is missing]
```

Without its markers, the bad message fails five of the code checks. The others, such as the greeting, the jargon and the hedging, need a judge:

In [ ]:
res = check(EXAMPLES[0]['record'], re.sub(r'\[[^\]]*\]', '', BAD))
test_eq({k for k, v in res.items() if not v}, {'format', 'coverage', 'severity', 'no_numbers', 'amount'})
res

{'format': False,
 'coverage': False,
 'no_invented': True,
 'severity': False,
 'no_numbers': False,
 'length': True,
 'amount': False}

## Failure modes

Each failure mode has a short name, what goes wrong, and the fix. `checks` names the code checks that catch it, and is empty when a judge must decide.

In [ ]:
#| export
FAILURES = [
    dict(name='chat', checks=(), what='A greeting, a sign-off or small talk.', fix='Start with the first item and stop after the last.'),
    dict(name='system-talk', checks=(), what='Talk about the monitoring, the analysis or the data.', fix='Say what is wrong with the plants or the sensors.'),
    dict(name='jargon', checks=(), what='Words from the record or from engineering, such as node, cluster, severity, flagged or implausible values.', fix='Use the plain words listed below.'),
    dict(name='numbers', checks=('no_numbers',), what='Digits outside the locations: readings, counts, percentages or dates.', fix="Use the amount words and the record's `since` phrase."),
    dict(name='format', checks=('format',), what='A paragraph that does not start with a marker, the location and a colon, or two items in one paragraph.', fix='One paragraph per item, each starting with its marker and its location.'),
    dict(name='marker', checks=('severity',), what="A marker that does not match the item's severity.", fix='👀 early, ⚠️ moderate, 🚨 severe.'),
    dict(name='missing-or-invented', checks=('coverage', 'no_invented'), what='An item of the record left out, or a location not in the record.', fix='Write exactly one paragraph for each item.'),
    dict(name='repeated-location', checks=(), what='The location, or part of it, repeated after the colon.', fix='Refer to it as this line, this plot, this plant.'),
    dict(name='hedging', checks=(), what='Likely, probably, may, might, possibly, it seems, about the problem or its cause.', fix='The code has decided. State the problem and its cause plainly.'),
    dict(name='vague-action', checks=(), what='No action, or one that does not follow from what the record reports.', fix='One concrete place to look, such as along the line when no water reaches the soil of a cluster.'),
    dict(name='wrong-amount', checks=('amount',), what="An amount word other than the record's `amount`, or none.", fix="Use the record's `amount` word as it is."),
    dict(name='wrong-tone', checks=(), what='Alarm for an early sign, or calm for a severe one.', fix='Match the urgency to the marker: say "today" for severe items.'),
    dict(name='too-long', checks=('length',), what='A message of more than 600 characters.', fix='One or two short sentences per item.'),
    dict(name='invented-cause', checks=(), what='A cause or part the record does not report, such as a stuck valve, an empty battery or a lost radio link.', fix='Say what the record reports, and where to look, never what broke.')]

The failure modes, numbered as in the bad message:

In [ ]:
Markdown(md_table([dict(f, n=i+1, checks=', '.join(f['checks'])) for i, f in enumerate(FAILURES)], ['n', 'name', 'what', 'fix', 'checks']))

n | name | what | fix | checks
--- | --- | --- | --- | ---
1 | chat | A greeting, a sign-off or small talk. | Start with the first item and stop after the last. | 
2 | system-talk | Talk about the monitoring, the analysis or the data. | Say what is wrong with the plants or the sensors. | 
3 | jargon | Words from the record or from engineering, such as node, cluster, severity, flagged or implausible values. | Use the plain words listed below. | 
4 | numbers | Digits outside the locations: readings, counts, percentages or dates. | Use the amount words and the record's `since` phrase. | no_numbers
5 | format | A paragraph that does not start with a marker, the location and a colon, or two items in one paragraph. | One paragraph per item, each starting with its marker and its location. | format
6 | marker | A marker that does not match the item's severity. | 👀 early, ⚠️ moderate, 🚨 severe. | severity
7 | missing-or-invented | An item of the record left out, or a location not in the record. | Write exactly one paragraph for each item. | coverage, no_invented
8 | repeated-location | The location, or part of it, repeated after the colon. | Refer to it as this line, this plot, this plant. | 
9 | hedging | Likely, probably, may, might, possibly, it seems, about the problem or its cause. | The code has decided. State the problem and its cause plainly. | 
10 | vague-action | No action, or one that does not follow from what the record reports. | One concrete place to look, such as along the line when no water reaches the soil of a cluster. | 
11 | wrong-amount | An amount word other than the record's `amount`, or none. | Use the record's `amount` word as it is. | amount
12 | wrong-tone | Alarm for an early sign, or calm for a severe one. | Match the urgency to the marker: say "today" for severe items. | 
13 | too-long | A message of more than 600 characters. | One or two short sentences per item. | length
14 | invented-cause | A cause or part the record does not report, such as a stuck valve, an empty battery or a lost radio link. | Say what the record reports, and where to look, never what broke. | 

The bad message shows every failure mode except wrong-tone and too-long, which would need a severe item or a longer record.

Every code check belongs to a failure mode:

In [ ]:
test_eq({c for f in FAILURES for c in f['checks']}, set(check(EXAMPLES[0]['record'], EXAMPLES[0]['message'])))

## Plain words

The record's vocabulary leaks into messages unless the prompt names the plain word for each term:

In [ ]:
#| export
PLAIN = {'node': 'sensor, or the plant it monitors', 'cluster': 'the plants on this line, or in this plot or bench',
    'widespread': 'across the farm', 'flagged': 'affected, or showing signs', 'missing data': 'sent no data',
    'implausible values': 'values that look wrong', 'camera view changed': 'the camera no longer sees the leaves',
    'severity, triage, pattern, soil_water, fault, sensor': 'leave these words out'}

In [ ]:
Markdown(md_table([dict(term=k, plain=v) for k, v in PLAIN.items()], ['term', 'plain']))

term | plain
--- | ---
node | sensor, or the plant it monitors
cluster | the plants on this line, or in this plot or bench
widespread | across the farm
flagged | affected, or showing signs
missing data | sent no data
implausible values | values that look wrong
camera view changed | the camera no longer sees the leaves
severity, triage, pattern, soil_water, fault, sensor | leave these words out

## The prompt

`system_prompt` assembles the parts in the order of write_prose: context, rules, good examples, the bad message, the failure modes, then plain words. `user_prompt` gives the record as JSON, and `prompt_version` identifies the prompt:

In [ ]:
#| export
def system_prompt()->str:
    "The teacher's system prompt"
    exs = '\n\n'.join(f"Record:\n{json.dumps(e['record'], ensure_ascii=False)}\n\nMessage:\n{e['message']}" for e in EXAMPLES)
    fails = '\n'.join(f"{i}. {f['what']} Fix: {f['fix']}" for i, f in enumerate(FAILURES, 1))
    plain = '\n'.join(f'- {k}: {v}' for k, v in PLAIN.items())
    return f"""{CONTEXT}

Rules:
{RULES}

Here are good messages for six records:

{exs}

Here is a bad message for the first record. Each bracketed number marks a mistake from the list that follows:

{BAD}

Mistakes to avoid:
{fails}

Plain words to use instead of the record's terms:
{plain}

Reply with the message only."""

def user_prompt(rec:dict)->str:
    "The teacher's prompt for one triage record"
    return json.dumps(rec, ensure_ascii=False)

def prompt_version()->str:
    "A short hash of the system prompt, which changes whenever the prompt does"
    return hashlib.sha256(system_prompt().encode()).hexdigest()[:12]

In [ ]:
sp = system_prompt()
len(sp), prompt_version(), sp[:300]

(9965,
 '3ec23e301454',
 'You write short WhatsApp messages to coffee farmers. Each message turns one triage record into actions. The record comes from monitoring code, which has already decided which problems matter, how severe they are, and what the data shows about their cause. Only some plants carry a sensor, and each se')

## Writing messages

`teach` sends the prompt and one record to the teacher through [fastllm](https://github.com/AnswerDotAI/fastllm), which also reaches other providers and llama.cpp's OpenAI-compatible server, so the baseline and fine-tuned models can go through the same client later. Each call gets a fresh chat, so records never see each other. A temperature of 0.3 keeps the messages consistent and still varies their wording. Haiku is the teacher, as the scope's Choice of teacher section explains, and `model` selects another one.

`teach` returns the message with its provenance, as the Choice of teacher section requires: the model, the temperature, the prompt version and the date. The prompt version is a short hash of the system prompt, so messages written with different prompts are never compared by mistake.

In [ ]:
#| exports
MODEL, TEMP = 'claude-haiku-4-5', 0.3

In [ ]:
#| export

async def teach(
    rec:dict, # Triage record
    model:str=MODEL, # Teacher model
    temp:float=TEMP, # Sampling temperature
    note:str='', # Added after the record, such as the mistakes of a previous attempt
)->dict: # The message, with the model, temperature, prompt version and date that produced it
    "Have the teacher write the message for `rec`"
    chat = AsyncChat(model, sp=system_prompt(), temp=temp, cache=True, cache_idxs=[0])
    r = await chat(user_prompt(rec) + (f'\n\n{note}' if note else ''), max_tokens=600)
    return dict(message=''.join(p.text for p in contents(r).content), model=model, temp=temp,
                prompt=prompt_version(), date=datetime.now(timezone.utc).isoformat(timespec='seconds'))

Calling the teacher needs `ANTHROPIC_API_KEY`, which `load_dotenv` reads from the project's untracked `.env` file. This cell does not run in tests:

In [ ]:
#| eval: false
from dotenv import load_dotenv
load_dotenv()
rec = sample(DIMS, triage, 10, seed=3)[8]['input']
await teach(rec)

For this record, with an implausible air temperature reading and two cameras whose view has changed, Haiku wrote:

> 👀 Block G, row 12: the air temperature sensor at this plant has given values that look wrong since today. Check that it is not damaged and that nothing blocks airflow around it.
>
> 👀 Block D, row 21: the camera at this plant has not seen its leaves since today. Check that it points at the plant and that nothing blocks its view.
>
> 👀 Block H, row 14: the camera at this plant has not seen its leaves since today. Check that it points at the plant and that nothing blocks its view.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()